# Speed (FP16 / TensorRT) + our models on new footage (about 1 hour, T4 GPU)

**Part 1, speed.** Our people model and ball model, three ways: plain, FP16, and a TensorRT FP16 engine.
Each fast version is checked against the plain one (same boxes?) and timed. The TensorRT engines are saved
next to the models in Drive, so the pipeline can use them (`MODEL_ACCEL=tensorrt`).

**Part 2, new footage.** Cuts five 20-second in-play clips from the full matches in
`MyDrive/Playbook/video_downloads/`: three from the Ittifaq match and two from match_video_2 (night), away from
the moments used for training. Runs our models on them (people + ball, keypoints through Roboflow every 3rd
frame) and saves everything, plus a small preview video, to `MyDrive/Playbook/new_footage/`. Claude replays
and scores them.

**Runtime:** T4 GPU. Colab Secret `ROBOFLOW_API_KEY` (keypoints; about 1,000 hosted calls).
**Runtime -> Run all**; do not edit cells. Other Google account: shared `Playbook` folder + shortcut in My Drive.

In [ ]:
# Cell 1 — GPU, Drive, paths
import os, sys, json, time, glob, shutil, subprocess
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')
print('GPU:', g.stdout.strip())
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
if not os.path.isdir(PBD):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
PEOPLE_CKPT = f'{PBD}/people_model/train/checkpoint_best_total.pth'
BALL_CKPT = f'{PBD}/ball_model/train/checkpoint_best_total.pth'
for p in (PEOPLE_CKPT, BALL_CKPT):
    if not os.path.exists(p):
        raise SystemExit(f'Missing model: {p}')
SRC_DIR = f'{PBD}/video_downloads'
OUT = f'{PBD}/new_footage'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Cell 2 — Latest repo + RF-DETR with TensorRT support
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
!pip install -q "rfdetr[tensorrt]==1.11.0"
try:
    import tensorrt
    print('TensorRT', tensorrt.__version__)
except Exception as e:
    print('TensorRT not available, the speed test will skip it:', e)

In [ ]:
# Cell 3 — Test frames (HILAL-HAZM clip) for the speed/parity check
import cv2, numpy as np
from vision.preprocess import enhance_frame
subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', 'final-deliverable'], cwd=PB, check=True)
clip = '/content/HILAL-HAZM_match_B_up10.mp4'
if not os.path.exists(clip):
    open(clip, 'wb').write(subprocess.run(['git', 'show', 'FETCH_HEAD:HILAL-HAZM_match_B_up10.mp4'], cwd=PB,
                                          capture_output=True, check=True).stdout)
cap = cv2.VideoCapture(clip); frames = []
for i in range(530):
    ok, fr = cap.read()
    if not ok: break
    if i % 26 == 0:
        frames.append(enhance_frame(fr)[:, :, ::-1].copy())   # RGB, as the pipeline feeds the models
cap.release()
TILE = 320
from tools.tile_dataset import grid_origins
tiles32 = [f[y:y + TILE, x:x + TILE] for f in frames[:4] for x, y in grid_origins(1920, 1080, TILE)][:32]
tiles4 = tiles32[:4]
print(len(frames), 'frames,', len(tiles32), 'tiles')

In [ ]:
# Cell 4 — Speed + agreement: plain vs FP16 vs TensorRT (engines saved next to the models in Drive)
import torch, gc
from rfdetr.detr import RFDETR
from vision.fast_rfdetr import accelerate_fp16, build_tensorrt_engine, TensorRTModel, compare_detections

def timeit(fn, n=20):
    fn(); torch.cuda.synchronize(); t = time.time()
    for _ in range(n): fn()
    torch.cuda.synchronize(); return 1000 * (time.time() - t) / n

speed = {}
for name, ckpt, thr, batches in [('people', PEOPLE_CKPT, 0.3, {'1 frame': frames[:1]}),
                                 ('ball', BALL_CKPT, 0.1, {'4 tiles': tiles4, '32 tiles': tiles32})]:
    res = speed[name] = {}
    plain = RFDETR.from_checkpoint(ckpt, trust_checkpoint=True)
    test_in = frames if name == 'people' else tiles32
    ref = plain.predict(list(test_in), threshold=thr)
    res['plain'] = {b: timeit(lambda x=x: plain.predict(list(x), threshold=thr)) for b, x in batches.items()}
    del plain; gc.collect(); torch.cuda.empty_cache()
    try:
        m = accelerate_fp16(RFDETR.from_checkpoint(ckpt, trust_checkpoint=True))
        out = m.predict(list(test_in), threshold=thr)
        res['fp16'] = {b: timeit(lambda x=x: m.predict(list(x), threshold=thr)) for b, x in batches.items()}
        res['fp16']['agreement'] = compare_detections(ref, out)
        del m
    except Exception as e:
        res['fp16'] = {'error': repr(e)[:300]}
    gc.collect(); torch.cuda.empty_cache()
    try:
        gpu = torch.cuda.get_device_name(0).replace(' ', '_')
        engine_dir = ckpt[:-4] + f'_trt_{gpu}'
        engines = glob.glob(f'{engine_dir}/*.trt') + glob.glob(f'{engine_dir}/*.engine')
        if not engines:
            m = RFDETR.from_checkpoint(ckpt, trust_checkpoint=True)
            engines = [build_tensorrt_engine(m, '/content/trt_' + name, max_batch=4 if name == 'people' else 32)]
            os.makedirs(engine_dir, exist_ok=True)
            shutil.copy(engines[0], engine_dir); engines = [f'{engine_dir}/{os.path.basename(engines[0])}']
            del m
        best = None
        for bg in (-1, None):   # which output slot is "background" differs by export; pick the one matching plain
            t = TensorRTModel(engines[0], background_class_id=bg)
            agree = compare_detections(ref, t.predict(list(test_in), threshold=thr))
            if best is None or agree['found'] > best[1]['found']:
                best = (bg, agree, t)
        bg, agree, t = best
        res['tensorrt'] = {b: timeit(lambda x=x: t.predict(list(x), threshold=thr)) for b, x in batches.items()}
        res['tensorrt'].update({'agreement': agree, 'background_class_id': bg, 'engine': engines[0]})
    except Exception as e:
        res['tensorrt'] = {'error': repr(e)[:500]}
    gc.collect(); torch.cuda.empty_cache()
    print(name, json.dumps(res, indent=1, default=str))
json.dump(speed, open(f'{OUT}/speed.json', 'w'), indent=1, default=str)

In [ ]:
# Cell 5 — Five 20-second in-play clips from the full matches
from tools.frame_sampler import pitch_green_ratio
FF = shutil.which('ffmpeg') or 'ffmpeg'
def duration_s(path):
    cap = cv2.VideoCapture(path); n = cap.get(cv2.CAP_PROP_FRAME_COUNT); fps = cap.get(cv2.CAP_PROP_FPS) or 25; cap.release()
    return n / fps
def _grab(cap, t):
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000); ok, fr = cap.read()
    return cv2.resize(fr, (480, 270)) if ok else None
def in_play_start(path, t0, limit_s=600):
    # in play = mostly pitch AND things moving (a halftime/replay graphic over a still pitch is not)
    cap = cv2.VideoCapture(path)
    for t in range(int(t0), int(t0 + limit_s), 10):
        a, b = _grab(cap, t), _grab(cap, t + 3)
        if a is None or b is None:
            continue
        moving = float(np.mean(cv2.absdiff(cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)))) >= 3.0
        if pitch_green_ratio(a) >= 0.30 and moving:
            cap.release(); return t
    cap.release(); return int(t0)
targets = []
itt = f'{SRC_DIR}/Ittifaq_f_match_video.mp4'
if os.path.exists(itt):
    d = duration_s(itt); targets += [('ittifaq_a', itt, 0.25 * d), ('ittifaq_b', itt, 0.50 * d), ('ittifaq_c', itt, 0.75 * d)]
mv2 = f'{SRC_DIR}/match_video_2.mp4'
if os.path.exists(mv2):
    targets += [('match2_a', mv2, 60 * 60), ('match2_b', mv2, 75 * 60)]   # far from the training clips (38-44, 90 min)
CLIPS = {}
for name, src, t0 in targets:
    dst = f'/content/clips/{name}.mp4'; os.makedirs('/content/clips', exist_ok=True)
    if not os.path.exists(dst):
        t = in_play_start(src, t0)
        subprocess.run([FF, '-y', '-loglevel', 'error', '-ss', str(t), '-i', src, '-t', '20', '-an',
                        '-c:v', 'libx264', '-crf', '18', '-preset', 'veryfast', dst], check=True)
        json.dump({'source': os.path.basename(src), 'start_s': t}, open(dst + '.json', 'w'))
    CLIPS[name] = dst
    print(name, json.load(open(dst + '.json')))

In [ ]:
# Cell 6 — Our models on each clip -> replay caches + preview videos in Drive
from google.colab import userdata
from tools.record_models import record, HostedModel
from vision.people_model import PeopleModel
from vision.ball_model import predict_tiles, rfdetr_predict_fn
from vision.fast_rfdetr import load_rfdetr
def ok_mode(r):   # fastest mode that reproduces the plain model
    for mode in ('tensorrt', 'fp16'):
        a = r.get(mode, {}).get('agreement') or {}
        if a.get('found', 0) >= 0.97 and a.get('extra_boxes', 1e9) <= 0.05 * max(a.get('ref_boxes', 1), 1):
            return mode
    return 'none'
people_mode, ball_mode = ok_mode(speed['people']), ok_mode(speed['ball'])
print('using people:', people_mode, '| ball:', ball_mode)
bg = lambda n, mode: speed[n]['tensorrt'].get('background_class_id', -1) if mode == 'tensorrt' else -1
people = PeopleModel(load_rfdetr(PEOPLE_CKPT, people_mode, max_batch=4, background_class_id=bg('people', people_mode)))
ball = rfdetr_predict_fn(load_rfdetr(BALL_CKPT, ball_mode, max_batch=32, background_class_id=bg('ball', ball_mode)))
ball_full = lambda img: predict_tiles(ball, img, tile=TILE, conf=0.10)
field = HostedModel('football-field-detection-f07vi-it2xv/10', userdata.get('ROBOFLOW_API_KEY'), keypoints=True)
for name, path in CLIPS.items():
    dst = f'{OUT}/{name}'
    if os.path.exists(f'{dst}/open/meta.json'):
        print(name, 'already done'); continue
    t0 = time.time()
    s = record(path, dst, {'open': (people, f'open-rfdetr-medium ({people_mode})')}, field,
               'football-field-detection-f07vi-it2xv/10', det_conf=0.10, field_every=3, ball_fn=ball_full, log_every=0)
    subprocess.run([FF, '-y', '-loglevel', 'error', '-i', path, '-vf', 'scale=960:-2', '-c:v', 'libx264',
                    '-crf', '30', '-preset', 'veryfast', f'{dst}/proxy_960.mp4'], check=True)
    shutil.copy(path + '.json', f'{dst}/clip.json')
    print(f"{name}: {s['frames']} frames in {(time.time() - t0) / 60:.1f} min, people {s['open']['ms_per_frame_median']:.0f} ms, "
          f"ball {s.get('ball_model', {}).get('ms_per_frame_median', 0):.0f} ms per frame")
print('\nDone. Tell Claude the results are in MyDrive/Playbook/new_footage/.')